您可以從 [Bookshop.org](https://bookshop.org/a/98697/9781098155438) 和 [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) 訂購《Think Python 3e》的印刷版和電子書版本。

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

# 文字分析與生成

目前我們已經涵蓋了 Python 的核心資料結構——串列、字典和元組——以及一些使用它們的演算法。
在本章中，我們將使用它們來探索文字分析和馬可夫生成：

* 文字分析是一種描述文件中單詞之間統計關係的方法，例如一個單詞被另一個單詞跟隨的機率，以及

* 馬可夫生成是一種產生新文字的方法，其中的單詞和短語類似於原始文字。

這些演算法類似於大型語言模型（LLM）的部分，這是聊天機器人的關鍵元件。

我們將從計算書中每個單詞出現的次數開始。
然後我們將查看單詞對，並列出可以跟隨每個單詞的單詞。
我們將製作馬可夫生成器的簡單版本，作為練習，你將有機會製作更通用的版本。

## 獨特單詞

作為文字分析的第一步，讓我們讀一本書——Robert Louis Stevenson 的《化身博士》——並計算獨特單詞的數量。
下載書籍的說明在本章的筆記本中。

以下單元格從古騰堡計畫下載這本書。

In [3]:
download('https://www.gutenberg.org/cache/epub/43/pg43.txt');

古騰堡計畫提供的版本在開頭包含關於書籍的資訊，在結尾包含授權資訊。
我們將使用第八章的 `clean_file` 來移除這些材料，並撰寫一個只包含書籍文字的「乾淨」檔案。

In [4]:
def is_special_line(line):
    return line.strip().startswith('*** ')

In [5]:
def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)
        
    reader.close()
    writer.close()

In [6]:
filename = 'dr_jekyll.txt'

In [7]:
clean_file('pg43.txt', filename)

我們將使用 `for` 迴圈從檔案中讀取行，並使用 `split` 將行分割成單詞。
然後，為了追蹤獨特單詞，我們將每個單詞作為鍵儲存在字典中。

In [9]:
unique_words = {}
for line in open(filename):
    seq = line.split()
    for word in seq:
        unique_words[word] = 1

len(unique_words)

6040

字典的長度就是獨特單詞的數量——按這種計算方式大約是 `6000` 個。
但如果我們檢查它們，我們會看到有些不是有效的單詞。

例如，讓我們看看 `unique_words` 中最長的單詞。
我們可以使用 `sorted` 來排序單詞，傳遞 `len` 函數作為關鍵字參數，以便按長度排序單詞。

In [88]:
sorted(unique_words, key=len)[-5:]

['chocolate-coloured',
 'superiors—behold!”',
 'coolness—frightened',
 'gentleman—something',
 'pocket-handkerchief.']

切片索引 `[-5:]` 選擇排序串列的最後 `5` 個元素，也就是最長的單詞。

串列包含一些合法的長單詞，如「circumscription」，以及一些用連字號連接的單詞，如「chocolate-coloured」。
但最長的一些「單詞」實際上是用破折號分隔的兩個單詞。
其他單詞包含標點符號，如句點、驚嘆號和引號。

因此，在我們繼續之前，讓我們處理破折號和其他標點符號。

## 標點符號

要識別文字中的單詞，我們需要處理兩個問題：

* 當破折號出現在行中時，我們應該用空格替換它——然後當我們使用 `split` 時，單詞會被分離。

* 分割單詞後，我們可以使用 `strip` 來移除標點符號。

為了處理第一個問題，我們可以使用以下函數，它接受字串，用空格替換破折號，分割字串，並回傳結果串列。

In [11]:
def split_line(line):
    return line.replace('—', ' ').split()

注意 `split_line` 只替換破折號，不替換連字號。
這是一個例子。

In [12]:
split_line('coolness—frightened')

['coolness', 'frightened']

現在，要從每個單詞的開頭和結尾移除標點符號，我們可以使用 `strip`，但我們需要被視為標點符號的字符串列。

Python 字串中的字符是 Unicode，這是一個國際標準，用於表示幾乎每個字母表中的字母、數字、符號、標點符號等。
`unicodedata` 模組提供了一個 `category` 函數，我們可以用它來辨別哪些字符是標點符號。
給定一個字母，它回傳一個包含該字母所屬類別資訊的字串。

In [13]:
import unicodedata

unicodedata.category('A')

'Lu'

`'A'` 的類別字串是 `'Lu'`——`'L'` 表示它是字母，`'u'` 表示它是大寫。

`'.'` 的類別字串是 `'Po'`——`'P'` 表示它是標點符號，`'o'` 表示其子類別是「其他」。

In [14]:
unicodedata.category('.')

'Po'

我們可以透過檢查類別以 `'P'` 開頭的字符來找到書中的標點符號。
以下迴圈將獨特的標點符號儲存在字典中。

In [15]:
punc_marks = {}
for line in open(filename):
    for char in line:
        category = unicodedata.category(char)
        if category.startswith('P'):
            punc_marks[char] = 1

要製作標點符號串列，我們可以將字典的鍵連接成字串。

In [16]:
punctuation = ''.join(punc_marks)
print(punctuation)

.’;,-“”:?—‘!()_


現在我們知道書中哪些字符是標點符號，我們可以撰寫一個函數，它接受單詞，從開頭和結尾去除標點符號，並轉換為小寫。

In [17]:
def clean_word(word):
    return word.strip(punctuation).lower()

這是一個例子。

In [18]:
clean_word('“Behold!”')

'behold'

因為 `strip` 從開頭和結尾移除字符，它會保留連字號單詞不變。

In [19]:
clean_word('pocket-handkerchief')

'pocket-handkerchief'

現在這是一個使用 `split_line` 和 `clean_word` 來識別書中獨特單詞的迴圈。

In [20]:
unique_words2 = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        unique_words2[word] = 1

len(unique_words2)

4005

使用這種更嚴格的單詞定義，大約有 4000 個獨特單詞。
我們可以確認最長單詞的串列已經被清理了。

In [21]:
sorted(unique_words2, key=len)[-5:]

['circumscription',
 'unimpressionable',
 'fellow-creatures',
 'chocolate-coloured',
 'pocket-handkerchief']

現在讓我們看看每個單詞使用了多少次。

## 單詞頻率

以下迴圈計算每個獨特單詞的頻率。

In [22]:
word_counter = {}
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        if word not in word_counter:
            word_counter[word] = 1
        else:
            word_counter[word] += 1

第一次看到單詞時，我們將其頻率初始化為 `1`。如果稍後再次看到同一個單詞，我們增加其頻率。

要查看哪些單詞最常出現，我們可以使用 `items` 從 `word_counter` 取得鍵值對，並按對的第二個元素（頻率）排序。
首先我們定義一個選擇第二個元素的函數。

In [23]:
def second_element(t):
    return t[1]

現在我們可以使用 `sorted` 搭配兩個關鍵字參數：

* `key=second_element` 表示項目將根據單詞的頻率排序。

* `reverse=True` 表示項目將以反向順序排序，最頻繁的單詞在前。

In [24]:
items = sorted(word_counter.items(), key=second_element, reverse=True)

這是五個最頻繁的單詞。

In [25]:
for word, freq in items[:5]:
    print(freq, word, sep='\t')

1614	the
972	and
941	of
640	to
640	i


在下一節中，我們將把這個迴圈封裝在函數中。
我們將使用它來展示一個新功能——可選參數。

## 可選參數

我們已經使用過接受可選參數的內建函數。
例如，`round` 接受一個名為 `ndigits` 的可選參數，指示要保留多少位小數。

In [26]:
round(3.141592653589793, ndigits=3)

3.142

但不只是內建函數——我們也可以撰寫有可選參數的函數。
例如，以下函數接受兩個參數，`word_counter` 和 `num`。

In [27]:
def print_most_common(word_counter, num=5):
    items = sorted(word_counter.items(), key=second_element, reverse=True)

    for word, freq in items[:num]:
        print(freq, word, sep='\t')

第二個參數看起來像指派陳述式，但不是——它是可選參數。

如果你用一個參數呼叫這個函數，`num` 會取得**預設值**，也就是 `5`。

In [28]:
print_most_common(word_counter)

1614	the
972	and
941	of
640	to
640	i


如果你用兩個參數呼叫這個函數，第二個參數會指派給 `num`，而不是預設值。

In [29]:
print_most_common(word_counter, 3)

1614	the
972	and
941	of


在這種情況下，我們會說可選參數**覆蓋**了預設值。

如果函數有必需參數和可選參數，所有必需參數必須在前，接著是可選參數。

In [30]:
%%expect SyntaxError

def bad_function(n=5, word_counter):
    return None

SyntaxError: non-default argument follows default argument (3116647453.py, line 1)

## 字典減法

假設我們想要對一本書進行拼字檢查——也就是找出可能拼錯的單詞串列。
一種方法是找出書中出現但不在有效單詞串列中的單詞。
在前面的章節中，我們使用過在 Scrabble 等文字遊戲中被認為有效的單詞串列。
現在我們將使用這個串列來對 Robert Louis Stevenson 進行拼字檢查。

我們可以將這個問題視為集合減法——也就是我們想要找到一個集合（書中的單詞）中不在另一個集合（串列中的單詞）中的所有單詞。

以下單元格下載單詞串列。

In [31]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

如前所述，我們可以讀取 `words.txt` 的內容並將其分割成字串串列。

In [32]:
word_list = open('words.txt').read().split()

然後我們將把單詞作為鍵儲存在字典中，以便可以使用 `in` 運算子快速檢查單詞是否有效。

In [33]:
valid_words = {}
for word in word_list:
    valid_words[word] = 1

現在，要識別出現在書中但不在單詞串列中的單詞，我們將使用 `subtract`，它接受兩個字典作為參數，並回傳一個新字典，包含一個字典中不在另一個字典中的所有鍵。

In [34]:
def subtract(d1, d2):
    res = {}
    for key in d1:
        if key not in d2:
            res[key] = d1[key]
    return res

這是我們如何使用它。

In [35]:
diff = subtract(word_counter, valid_words)

要取得可能拼錯的單詞樣本，我們可以列印 `diff` 中最常見的單詞。

In [36]:
print_most_common(diff)

640	i
628	a
128	utterson
124	mr
98	hyde


最常見的「拼錯」單詞主要是姓名和一些單字母單詞（Utterson 先生是 Jekyll 博士的朋友和律師）。

如果我們選擇只出現一次的單詞，它們更可能是實際的拼寫錯誤。
我們可以透過迴圈遍歷項目並製作頻率為 `1` 的單詞串列來做到這一點。

In [37]:
singletons = []
for word, freq in diff.items():
    if freq == 1:
        singletons.append(word)

這是串列的最後幾個元素。

In [38]:
singletons[-5:]

['gesticulated', 'abjection', 'circumscription', 'reindue', 'fearstruck']

大部分都是不在單詞串列中的有效單詞。
但 `'reindue'` 似乎是 `'reinduce'` 的拼寫錯誤，所以至少我們找到了一個合法的錯誤。

## 隨機數

作為邁向馬可夫文字生成的一步，接下來我們將從 `word_counter` 中選擇隨機的單詞序列。
但首先讓我們談談隨機性。

給定相同的輸入，大多數電腦程式是**確定性的**，這意味著它們每次都產生相同的輸出。
確定性通常是好事，因為我們期望相同的計算產生相同的結果。
但對於某些應用程式，我們希望電腦是不可預測的。
遊戲是一個例子，但還有更多。

使程式真正非確定性變得困難，但有方法可以偽造它。
其中一種方法是使用生成**偽隨機**數的演算法。
偽隨機數不是真正隨機的，因為它們是由確定性計算生成的，但僅僅透過查看數字，幾乎不可能將它們與隨機數區分開來。

`random` 模組提供生成偽隨機數的函數——我在這裡將簡稱為「隨機」。
我們可以像這樣匯入它。

In [39]:
import random

In [40]:
# this cell initializes the random number generator so it 
# generates the same sequence each time the notebook runs.

random.seed(4)

`random` 模組提供一個名為 `choice` 的函數，它從串列中隨機選擇一個元素，每個元素被選擇的機率相同。

In [41]:
t = [1, 2, 3]
random.choice(t)

1

如果你再次呼叫這個函數，你可能再次得到相同的元素，或者得到不同的元素。

In [42]:
random.choice(t)

2

從長遠來看，我們期望每個元素出現的次數大致相同。

如果你對字典使用 `choice`，你會得到 `KeyError`。

In [43]:
%%expect KeyError

random.choice(word_counter)

KeyError: 422

要選擇隨機鍵，你必須將鍵放入串列中，然後呼叫 `choice`。

In [44]:
words = list(word_counter)
random.choice(words)

'posture'

如果我們生成隨機單詞序列，它沒有太大意義。

In [45]:
for i in range(6):
    word = random.choice(words)
    print(word, end=' ')

ill-contained written apocryphal nor busy spoke 

問題的一部分是我們沒有考慮到某些單詞比其他單詞更常見。
如果我們使用不同的「權重」來選擇單詞，使得某些單詞被選擇得更頻繁，結果會更好。

如果我們使用 `word_counter` 的值作為權重，每個單詞被選擇的機率取決於其頻率。

In [46]:
weights = word_counter.values()

`random` 模組提供另一個名為 `choices` 的函數，它接受權重作為可選參數。

In [47]:
random.choices(words, weights=weights)

['than']

它還接受另一個可選參數 `k`，指定要選擇的單詞數量。

In [48]:
random_words = random.choices(words, weights=weights, k=6)
random_words

['reach', 'streets', 'edward', 'a', 'said', 'to']

結果是字串串列，我們可以將它們連接成看起來更像句子的東西。

In [49]:
' '.join(random_words)

'reach streets edward a said to'

如果你從書中隨機選擇單詞，你會對詞彙有所感受，但隨機單詞的序列很少有意義，因為連續單詞之間沒有關係。
例如，在真正的句子中，你期望像「the」這樣的冠詞後面跟著形容詞或名詞，而可能不是動詞或副詞。
所以下一步是查看這些單詞之間的關係。

## 雙字元組

現在我們不再一次看一個單詞，而是要看兩個單詞的序列，稱為**雙字元組**。
三個單詞的序列稱為**三字元組**，具有某些未指定單詞數的序列稱為 **n 字元組**。

讓我們撰寫一個程式，找到書中所有的雙字元組以及每個雙字元組出現的次數。
為了儲存結果，我們將使用一個字典，其中

* 鍵是代表雙字元組的字串元組，以及

* 值是代表頻率的整數。

讓我們稱它為 `bigram_counter`。

In [50]:
bigram_counter = {}

以下函數接受兩個字串的串列作為參數。
首先它製作兩個字串的元組，這可以用作字典中的鍵。
然後它將鍵新增到 `bigram_counter`，如果不存在的話，或者如果存在則增加頻率。

In [51]:
def count_bigram(bigram):
    key = tuple(bigram)
    if key not in bigram_counter:
        bigram_counter[key] = 1
    else:
        bigram_counter[key] += 1

當我們翻閱書籍時，我們必須追蹤每對連續的單詞。
所以如果我們看到序列「man is not truly one」，我們會新增雙字元組「man is」、「is not」、「not truly」等等。

為了追蹤這些雙字元組，我們將使用一個名為 `window` 的串列，因為它就像一個在書頁上滑動的窗口，一次只顯示兩個單詞。
最初，`window` 是空的。

In [52]:
window = []

我們將使用以下函數一次處理一個單詞。

In [53]:
def process_word(word):
    window.append(word)
    
    if len(window) == 2:
        count_bigram(window)
        window.pop(0)

第一次呼叫這個函數時，它將給定的單詞附加到 `window`。
由於窗口中只有一個單詞，我們還沒有雙字元組，所以函數結束。

第二次呼叫時——以及之後的每次——它將第二個單詞附加到 `window`。
由於窗口中有兩個單詞，它呼叫 `count_bigram` 來追蹤每個雙字元組出現的次數。
然後它使用 `pop` 從窗口中移除第一個單詞。

以下程式迴圈遍歷書中的單詞並一次處理一個。

In [54]:
for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word(word)

結果是一個從每個雙字元組對應到它出現次數的字典。
我們可以使用 `print_most_common` 來查看最常見的雙字元組。

In [55]:
print_most_common(bigram_counter)

178	('of', 'the')
139	('in', 'the')
94	('it', 'was')
80	('and', 'the')
73	('to', 'the')


看到這些結果，我們可以感受到哪些單詞對最可能一起出現。
我們也可以使用結果來生成隨機文字，像這樣。

In [56]:
random.seed(0)

In [57]:
bigrams = list(bigram_counter)
weights = bigram_counter.values()
random_bigrams = random.choices(bigrams, weights=weights, k=6)

`bigrams` 是書中出現的雙字元組串列。
`weights` 是它們的頻率串列，所以 `random_bigrams` 是一個樣本，其中雙字元組被選擇的機率與其頻率成正比。

結果如下。

In [58]:
for pair in random_bigrams:
    print(' '.join(pair), end=' ')

to suggest this preface to detain fact is above all the laboratory 

這種生成文字的方式比選擇隨機單詞要好，但仍然沒有太大意義。

## 馬可夫分析

我們可以用馬可夫鏈文字分析做得更好，它計算文字中每個單詞的後續單詞串列。
作為例子，我們將分析 Monty Python 歌曲《Eric, the Half a Bee》的這些歌詞：

In [59]:
song = """
Half a bee, philosophically,
Must, ipso facto, half not be.
But half the bee has got to be
Vis a vis, its entity. D'you see?
"""

要儲存結果，我們將使用一個從每個單詞對應到跟隨它的單詞串列的字典。

In [60]:
successor_map = {}

作為例子，讓我們從歌曲的前兩個單詞開始。

In [61]:
first = 'half'
second = 'a'

如果第一個單詞不在 `successor_map` 中，我們必須新增一個新項目，從第一個單詞對應到包含第二個單詞的串列。

In [62]:
successor_map[first] = [second]
successor_map

{'half': ['a']}

如果第一個單詞已經在字典中，我們可以查找它來獲得到目前為止看到的後續單詞串列，並附加新的。

In [63]:
first = 'half'
second = 'not'

successor_map[first].append(second)
successor_map

{'half': ['a', 'not']}

以下函數封裝了這些步驟。

In [64]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

如果相同的雙字元組出現超過一次，第二個單詞會被多次新增到串列中。
這樣，`successor_map` 記錄每個後續單詞出現多少次。

如前一節所述，我們將使用名為 `window` 的串列來儲存連續的單詞對。
我們將使用以下函數一次處理一個單詞。

In [65]:
def process_word_bigram(word):
    window.append(word)
    
    if len(window) == 2:
        add_bigram(window)
        window.pop(0)

這是我們如何使用它來處理歌曲中的單詞。

In [66]:
successor_map = {}
window = []

for word in song.split():
    word = clean_word(word)
    process_word_bigram(word)

結果如下。

In [67]:
successor_map

{'half': ['a', 'not', 'the'],
 'a': ['bee', 'vis'],
 'bee': ['philosophically', 'has'],
 'philosophically': ['must'],
 'must': ['ipso'],
 'ipso': ['facto'],
 'facto': ['half'],
 'not': ['be'],
 'be': ['but', 'vis'],
 'but': ['half'],
 'the': ['bee'],
 'has': ['got'],
 'got': ['to'],
 'to': ['be'],
 'vis': ['a', 'its'],
 'its': ['entity'],
 'entity': ["d'you"],
 "d'you": ['see']}

單詞 `'half'` 可以跟隨 `'a'`、`'not'` 或 `'the'`。
單詞 `'a'` 可以跟隨 `'bee'` 或 `'vis'`。
大多數其他單詞只出現一次，所以它們只跟隨一個單詞。

現在讓我們分析這本書。

In [68]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_bigram(word)

我們可以查找任何單詞並找到可以跟隨它的單詞。

In [69]:
# I used this cell to find a predecessor with a good number of possible successors
# and at least one repeated word.

def has_duplicates(t):
    return len(set(t)) < len(t)

for key, value in successor_map.items():
    if len(value) == 7 and has_duplicates(value):
        print(key, value)

story ['of', 'of', 'indeed', 'but', 'for', 'of', 'that']
incident ['of', 'of', 'at', 'of', 'of', 'at', 'this']
lanyon’s ['narrative', 'there', 'face', 'manner', 'narrative', 'the', 'condemnation']
common ['it', 'interest', 'friends', 'friends', 'observers', 'but', 'quarry']
relief ['the', 'to', 'when', 'that', 'that', 'of', 'it']
appearance ['of', 'well', 'something', 'he', 'amply', 'of', 'which']
going ['east', 'in', 'to', 'to', 'up', 'to', 'of']
till ['at', 'the', 'i', 'yesterday', 'the', 'that', 'weariness']
walk ['and', 'into', 'with', 'all', 'steadfastly', 'attired', 'with']
sounds ['nothing', 'carried', 'out', 'the', 'of', 'of', 'with']
really ['like', 'damnable', 'can', 'a', 'a', 'not', 'be']
does ['not', 'not', 'indeed', 'not', 'the', 'not', 'not']
reply ['i', 'but', 'whose', 'i', 'some', 'that’s', 'i']
continued ['mr', 'the', 'the', 'the', 'the', 'poole', 'utterson']
seems ['scarcely', 'hardly', 'to', 'she', 'much', 'he', 'to']
walked ['on', 'over', 'some', 'was', 'on', 'with'

In [70]:
successor_map['going']

['east', 'in', 'to', 'to', 'up', 'to', 'of']

在這個後續單詞串列中，注意單詞 `'to'` 出現三次——其他後續單詞只出現一次。

## 生成文字

我們可以使用前一節的結果來生成新文字，其中連續單詞之間的關係與原文相同。
運作方式如下：

* 從文字中出現的任何單詞開始，我們查找其可能的後續單詞並隨機選擇一個。

* 然後，使用選擇的單詞，我們查找其可能的後續單詞，並隨機選擇一個。

我們可以重複這個過程來生成我們想要的任意數量的單詞。
作為例子，讓我們從單詞 `'although'` 開始。
這是可以跟隨它的單詞。

In [71]:
word = 'although'
successors = successor_map[word]
successors

['i', 'a', 'it', 'the', 'we', 'they', 'i']

In [72]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(2)

我們可以使用 `choice` 以相等機率從串列中選擇。

In [73]:
word = random.choice(successors)
word

'i'

如果同一個單詞在串列中出現超過一次，它更可能被選擇。

重複這些步驟，我們可以使用以下迴圈來生成更長的序列。

In [74]:
for i in range(10):
    successors = successor_map[word]
    word = random.choice(successors)
    print(word, end=' ')

continue to hesitate and swallowed the smile withered from that 

結果聽起來更像真正的句子，但仍然沒有太大意義。

我們可以使用超過一個單詞作為 `successor_map` 中的鍵來做得更好。
例如，我們可以建立一個從每個雙字元組——或三字元組——對應到接下來單詞串列的字典。
作為練習，你將有機會實作這種分析並看看結果如何。

## 除錯

此時我們正在撰寫更實質的程式，你可能會發現花在除錯上的時間更多。
如果你遇到困難的錯誤，以下是一些嘗試的方法：

* 閱讀：檢查你的程式碼，讀給自己聽，並檢查它是否表達了你想要表達的意思。

* 執行：透過修改和執行不同版本來實驗。通常如果你在程式的正確位置顯示正確的東西，問題就變得明顯，但有時你需要建立基礎設施。

* 思考：花一些時間思考！這是什麼類型的錯誤：語法、執行時或語義？你能從錯誤訊息或程式輸出中獲得什麼資訊？什麼類型的錯誤可能導致你看到的問題？在問題出現之前，你最後修改了什麼？

* 橡皮鴨除錯：如果你向別人解釋問題，有時你會在問完問題之前就找到答案。通常你不需要另一個人；你可以只是對橡皮鴨說話。這就是著名策略「**橡皮鴨除錯**」的起源。我不是在開玩笑——參見 <https://en.wikipedia.org/wiki/Rubber_duck_debugging>。

* 撤退：在某些情況下，最好的做法是退後——撤銷最近的修改——直到你回到一個可運作的程式。然後你可以開始重建。
    
* 休息：如果你讓大腦休息一下，有時它會為你找到問題。

初學程式設計者有時會困在其中一項活動上，忘記其他活動。每項活動都有自己的失敗模式。

例如，如果問題是打字錯誤，閱讀程式碼會有效，但如果問題是概念性誤解則不會。
如果你不了解你的程式做什麼，你可以讀一百遍也永遠看不到錯誤，因為錯誤在你的腦中。

執行實驗可能有效，特別是如果你執行小而簡單的測試。
但如果你在沒有思考或閱讀程式碼的情況下執行實驗，可能需要很長時間才能弄清楚發生了什麼。

你必須花時間思考。除錯就像實驗科學。你應該對問題是什麼至少有一個假設。如果有兩個或更多可能性，試著想出一個可以排除其中一個的測試。

但即使是最好的除錯技術，如果有太多錯誤，或者你試圖修復的程式碼太大太複雜，也會失敗。
有時最好的選擇是撤退，簡化程式，直到你回到能運作的東西。

初學程式設計者通常不願意撤退，因為他們無法忍受刪除一行程式碼（即使它是錯的）。如果這讓你感覺好一些，在開始精簡之前將你的程式複製到另一個檔案。然後你可以一次一個地複製回去。

找到困難的錯誤需要閱讀、執行、思考、撤退，有時還需要休息。
如果你困在其中一項活動上，試試其他的。

## 術語解釋

**預設值：**
如果沒有提供參數時指派給參數的值。

**覆蓋：**
 用參數替換預設值。

**確定性：**
 確定性程式在給定相同輸入時，每次執行都做相同的事情。

**偽隨機：**
 偽隨機數字序列看起來是隨機的，但是由確定性程式生成的。

**雙字元組：**
兩個元素的序列，通常是單詞。

**三字元組：**
三個元素的序列。

**n 字元組：**
未指定數量元素的序列。

**橡皮鴨除錯：**
透過向無生物解釋問題來除錯的方法。

## 練習

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### 詢問虛擬助理

在 `add_bigram` 中，`if` 陳述式建立新串列或將元素附加到現有串列，取決於鍵是否已經在字典中。

In [75]:
def add_bigram(bigram):
    first, second = bigram
    
    if first not in successor_map:
        successor_map[first] = [second]
    else:
        successor_map[first].append(second)

字典提供一個名為 `setdefault` 的方法，我們可以用它更簡潔地做同樣的事情。
詢問虛擬助理它如何運作，或將 `add_word` 複製到虛擬助理並詢問「你能使用 `setdefault` 重寫這個嗎？」

在本章中我們實作了馬可夫鏈文字分析和生成。
如果你很好奇，你可以向虛擬助理詢問有關該主題的更多資訊。
你可能學到的一件事是虛擬助理使用在許多方面相似但在重要方面也不同的演算法。
詢問虛擬助理：「像 GPT 這樣的大型語言模型與馬可夫鏈文字分析有什麼區別？」

### 練習

撰寫一個函數，計算每個三字元組（三個單詞的序列）出現的次數。
如果你用《化身博士》的文字測試你的函數，你應該發現最常見的三字元組是「said the lawyer」。

提示：撰寫一個名為 `count_trigram` 的函數，類似於 `count_bigram`。然後撰寫一個名為 `process_word_trigram` 的函數，類似於 `process_word_bigram`。

In [76]:
# Solution

def count_trigram(trigram):
    key = tuple(trigram)
    
    if key not in trigram_counter:
        trigram_counter[key] = 1
    else:
        trigram_counter[key] += 1

In [77]:
# Solution

def process_word_trigram(word):
    window.append(word)
    
    if len(window) == 3:
        count_trigram(window)
        window.pop(0)

你可以使用以下迴圈來讀取書籍並處理單詞。

In [78]:
trigram_counter = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

然後使用 `print_most_common` 來找到書中最常見的三字元組。

In [79]:
print_most_common(trigram_counter)

20	('said', 'the', 'lawyer')
16	('said', 'mr', 'utterson')
15	('of', 'edward', 'hyde')
13	('the', 'name', 'of')
11	('it', 'was', 'a')


### 練習

現在讓我們實作馬可夫鏈文字分析，從每個雙字元組對應到可能後續單詞串列的對應。

從 `add_bigram` 開始，撰寫一個名為 `add_trigram` 的函數，它接受三個單詞的串列，並在 `successor_map` 中新增或更新項目，使用前兩個單詞作為鍵，第三個單詞作為可能的後續單詞。

In [80]:
# Solution

def add_trigram(trigram):
    first, second, third = trigram
    key = first, second
    
    if key not in successor_map:
        successor_map[key] = [word]
    else:
        successor_map[key].append(word)

這是呼叫 `add_trigram` 的 `process_word_trigram` 版本。

In [81]:
def process_word_trigram(word):
    window.append(word)
    
    if len(window) == 3:
        add_trigram(window)
        window.pop(0)

你可以使用以下迴圈來用「Eric, the Half a Bee」的歌詞測試你的函數。

In [82]:
successor_map = {}
window = []

for string in song.split():
    word = string.strip(punctuation).lower()
    process_word_trigram(word)

如果你的函數按預期運作，前綴 `('half', 'a')` 應該對應到只有單一元素 `'bee'` 的串列。
事實上，碰巧這首歌中的每個雙字元組只出現一次，所以 `successor_map` 中的所有值都只有一個元素。

In [83]:
successor_map

{('half', 'a'): ['bee'],
 ('a', 'bee'): ['philosophically'],
 ('bee', 'philosophically'): ['must'],
 ('philosophically', 'must'): ['ipso'],
 ('must', 'ipso'): ['facto'],
 ('ipso', 'facto'): ['half'],
 ('facto', 'half'): ['not'],
 ('half', 'not'): ['be'],
 ('not', 'be'): ['but'],
 ('be', 'but'): ['half'],
 ('but', 'half'): ['the'],
 ('half', 'the'): ['bee'],
 ('the', 'bee'): ['has'],
 ('bee', 'has'): ['got'],
 ('has', 'got'): ['to'],
 ('got', 'to'): ['be'],
 ('to', 'be'): ['vis'],
 ('be', 'vis'): ['a'],
 ('vis', 'a'): ['vis'],
 ('a', 'vis'): ['its'],
 ('vis', 'its'): ['entity'],
 ('its', 'entity'): ["d'you"],
 ('entity', "d'you"): ['see']}

你可以使用以下迴圈來用書中的單詞測試你的函數。

In [84]:
successor_map = {}
window = []

for line in open(filename):
    for word in split_line(line):
        word = clean_word(word)
        process_word_trigram(word)

在下一個練習中，你將使用結果來生成新的隨機文字。

### 練習

對於這個練習，我們假設 `successor_map` 是一個從每個雙字元組對應到跟隨它的單詞串列的字典。

In [85]:
# this cell initializes the random number generator so it 
# starts at the same point in the sequence each time this
# notebook runs.

random.seed(3)

要生成隨機文字，我們先從 `successor_map` 中選擇一個隨機鍵。

In [86]:
successors = list(successor_map)
bigram = random.choice(successors)
bigram

('doubted', 'if')

現在撰寫一個迴圈，按照以下步驟生成 50 個更多的單詞：

1. 在 `successor_map` 中，查找可以跟隨 `bigram` 的單詞串列。

2. 隨機選擇其中一個並列印它。

3. 對於下次迭代，建立一個新的雙字元組，包含 `bigram` 的第二個單詞和選擇的後續單詞。

例如，如果我們從雙字元組 `('doubted', 'if')` 開始並選擇 `'from'` 作為其後續單詞，下一個雙字元組是 `('if', 'from')`。

In [89]:
# Solution

for i in range(50):
    successors = successor_map[bigram]
    word = random.choice(successors)    
    print(word, end=' ')
    
    first, second = bigram
    bigram = second, word

was perhaps relieved to be born and at the horror of being hyde that racked me i have been about half full of premature twilight although the sky high up overhead was still rolling in through the by-street and that a mr hyde had numbered few familiars even the nightmares 

如果一切正常，你應該發現生成的文字在風格上與原文明顯相似，某些短語有意義，但文字可能會從一個主題漫遊到另一個主題。

作為獎勵練習，修改你對最後兩個練習的解決方案，使用三字元組作為 `successor_map` 中的鍵，並看看這對結果有什麼影響。

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

版權所有 2024 [Allen B. Downey](https://allendowney.com)

程式碼授權：[MIT License](https://mit-license.org/)

文字授權：[Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)